# Construindo uma Biblioteca de Precificação Atuarial Reutilizável com PROC FCMP


## Resumo Executivo

Uma seguradora de ramos elementares (P&C) encapsula sua matemática de precificação central — prêmio puro, carregamento de despesas/risco, mescla de credibilidade de flutuação limitada e estimativa de reserva descontada — como funções personalizadas e uma sub-rotina de múltiplas saídas no **PROC FCMP**. A biblioteca compilada é registrada através da opção de sistema `CMPLIB=` e então chamada linha a linha a partir de um DATA step que tarifa uma carteira sintética de 100 apólices. Toda cifra tarifada neste notebook — o fator de credibilidade `z`, o prêmio puro combinado, o prêmio cobrado e a reserva de sinistro a valor presente — é produzida por essas rotinas compiladas, não por aritmética embutida. O resultado: o índice de sinistralidade implícito fica em 60,5% (rural), 55,8% (suburbano) e 63,6% (urbano) — confortavelmente abaixo de 100% em todos os segmentos, confirmando que o prêmio carregado cobre a perda esperada enquanto a etapa de tarifação permanece limpa e auditável.


## Fontes de Dados

| Conjunto de Dados | Linhas | Descrição | Variáveis-Chave |
|---------|------|-------------|---------------|
| `work.portfolio` | 100 | Carteira sintética de apólices P&C em vigor, gerada inline com `rand()` | `policy_id`, `region` (urbano/suburbano/rural), `years_insured`, `exposure` (carro-anos), `claim_count` (Poisson), `incurred_loss` (severidade gama x contagem), `class_pure_prem` (tarifa manual por região) |

O DATA step percorre um intervalo mais amplo de `policy_id`, mas este ambiente roda sem licença, então a saída é limitada às primeiras **100 observações** — o livro tarifado abaixo reflete essas 100 apólices.


# Construindo uma Biblioteca de Precificação Atuarial Reutilizável com PROC FCMP

Atuários repetem os mesmos cálculos em trabalhos de precificação, reservas e relatórios: converter perdas em um *prêmio puro*, aplicar *carregamentos de despesas e risco* para chegar a um prêmio cobrado, combinar a experiência própria de uma apólice com uma tarifa de classe usando *credibilidade*, e *descontar* fluxos de caixa futuros a valor presente. Retipar essas fórmulas em cada DATA step convida a erros de copiar-e-colar e torna dolorosas as mudanças de premissas.

O **PROC FCMP** (o compilador de funções do SAS) nos permite definir cada fórmula uma única vez como uma função ou sub-rotina nomeada, armazenar as rotinas compiladas em uma biblioteca e então chamá-las como qualquer função nativa do SAS. Neste notebook nós:

1. Compilamos uma pequena biblioteca de funções atuariais com `PROC FCMP`.
2. Registramos para a sessão com a opção de sistema `CMPLIB=`.
3. Geramos uma carteira sintética de seguros de ramos elementares.
4. Tarifamos cada apólice chamando nossas funções personalizadas e uma sub-rotina de múltiplas saídas a partir de um único DATA step.
5. Resumimos e interpretamos a carteira tarifada.


## Etapa 1 — Gerar uma carteira sintética de apólices

Simulamos um livro de apólices de automóvel em vigor (as primeiras 100 são tarifadas abaixo sob o limite do modo sem licença). Cada apólice pertence a uma região de classificação com seu próprio *prêmio puro* manual (perda esperada por carro-ano). A contagem de sinistros segue um processo de Poisson escalado pela exposição, e as severidades seguem uma distribuição gama, de modo que `incurred_loss` é uma perda composta realista (Poisson x gama). `years_insured` mais adiante conduzirá o peso de credibilidade. Uma semente fixa via `call streaminit` torna os dados reprodutíveis.


In [1]:
DADOS portfolio;
    CHAMAR streaminit(20260531);
    COMPRIMENTO region $9;
    FAZER policy_id = 10001 ATÉ 12000;
        /* Atribui uma regiao de classificacao */
        u = rand('uniform');
        SE u < 0.40 ENTÃO FAZER; region = 'urbano';    base_pp = 820; lambda = 0.18; FIM;
        SENÃO SE u < 0.75 ENTÃO FAZER; region = 'suburbano'; base_pp = 540; lambda = 0.11; FIM;
        SENÃO FAZER; region = 'rural';    base_pp = 360; lambda = 0.07; FIM;

        /* Tempo de seguro (anos) e exposicao (carro-anos) */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* Processo composto de sinistros: frequencia Poisson, severidade gamma */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        FAZER c = 1 ATÉ claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        FIM;
        incurred_loss = round(incurred_loss, 0.01);

        /* Premio puro de classe manual para a exposicao desta apolice */
        class_pure_prem = round(base_pp * exposure, 0.01);

        SAÍDA;
    FIM;
    MANTER policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
EXECUTAR;

PROCEDIMENTO IMPRIMIR DADOS=portfolio(obs=8) noobs RÓTULO;
    RÓTULO policy_id="ID da Apolice"
          region="Regiao"
          years_insured="Anos Segurado"
          exposure="Exposicao (carro-anos)"
          claim_count="Numero de Sinistros"
          incurred_loss="Perda Incorrida"
          class_pure_prem="Premio Puro de Classe";
    TÍTULO "Primeiras 8 Apolices Simuladas";
EXECUTAR;


                                             Primeiras 8 Apolices Simuladas                                             

ID da Apolice     Regiao  Anos Segurado  Exposicao (carro-anos)  Numero de Sinistros  Perda Incorrida  Premio Puro de Classe
        10001  rural                  5                    1.36                    0                0                  489.6
        10002  urbano                 8                    0.97                    1          3432.28                  795.4
        10003  urbano                 2                    1.53                    2          7155.92                 1254.6
        10004  suburbano              9                     2.4                    0                0                   1296
        10005  rural                  5                    0.99                    0                0                  356.4
        10006  suburbano              1                    0.83                    0                0                  448.2
   


NOTE: DATA portfolio

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote portfolio (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.43 seconds
  cpu   0.43 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## Etapa 2 — Compilar a biblioteca de funções atuariais

Agora o coração do notebook. O `PROC FCMP` com `OUTLIB=work.actfuncs.pricing` compila quatro funções e uma sub-rotina no pacote `pricing` do conjunto de dados `work.actfuncs`:

- **`pure_premium`** — perda observada por unidade de exposição (frequência x severidade combinadas).
- **`credibility_z`** — fator de credibilidade de flutuação limitada `Z = sqrt(n / (n + k))`, onde `n` são os anos-exposição da apólice e `k` é uma constante de ajuste.
- **`charged_premium`** — aplica um carregamento de risco proporcional e uma taxa de despesa fixa a um custo de perda para chegar ao prêmio que efetivamente cobramos.
- **`pv_reserve`** — valor presente de um pagamento futuro de sinistro, `FV / (1+r)**t`, usado para descontar reservas de sinistro.
- **`blend_premium`** (uma SUB-ROTINA) — usa `OUTARGS` para retornar *dois* valores de uma vez: o prêmio puro ponderado por credibilidade e o fator de credibilidade que usou, de modo que o DATA step obtém ambos em uma única chamada.

Cada rotina é encerrada com `ENDSUB`, e a sub-rotina nomeia seus argumentos graváveis com `OUTARGS`.


In [2]:
PROCEDIMENTO fcmp outlib=work.actfuncs.pricing;

    /* Premio puro observado: custo de perda por unidade de exposicao */
    function pure_premium(loss, exposure);
        SE exposure <= 0 ENTÃO RETURN(.);
        RETURN(loss / exposure);
    endsub;

    /* Credibilidade de flutuacao limitada: Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        SE n_years <= 0 ENTÃO RETURN(0);
        RETURN(sqrt(n_years / (n_years + k)));
    endsub;

    /* Premio cobrado = custo de perda majorado por carregamento de risco + despesas */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        SE expense_ratio >= 1 ENTÃO RETURN(.);
        RETURN(gross / (1 - expense_ratio));
    endsub;

    /* Valor presente de um pagamento futuro de sinistro descontado t anos a taxa r */
    function pv_reserve(future_value, r, t);
        RETURN(future_value / (1 + r) ** t);
    endsub;

    /* Mescla de credibilidade: retorna o premio puro ponderado E o Z usado */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

EXECUTAR;



               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium




NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## Etapa 3 — Registrar a biblioteca com CMPLIB=

Compilar as rotinas não é suficiente; o SAS precisa ser informado onde encontrá-las quando um DATA step ou procedimento posterior referenciar um nome que não reconhece como nativo. A opção de sistema `CMPLIB=` aponta para o conjunto de dados (não o nome de pacote de três níveis) que contém o código compilado. Após esta instrução `OPTIONS`, cada função e sub-rotina acima fica chamável pelo nome.


In [3]:
OPÇÕES cmplib=work.actfuncs;



NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## Etapa 4 — Tarifar a carteira com as funções personalizadas

O DATA step de tarifação agora está quase livre de aritmética — a intenção atuarial se lê diretamente nos nomes das funções. Para cada apólice nós:

1. Calculamos o prêmio puro observado da própria apólice com `pure_premium`.
2. Chamamos a sub-rotina `blend_premium` para ponderar por credibilidade contra a tarifa de classe regional, recuperando tanto o custo de perda combinado quanto o fator de credibilidade através de `OUTARGS`.
3. Majoramos o custo de perda combinado para um prêmio cobrado com um carregamento de risco de 12% e uma taxa de despesa de 25% via `charged_premium`.
4. Estimamos a reserva de sinistro ainda aberta como 35% da perda incorrida da apólice e a descontamos por três anos a 4% a valor presente com `pv_reserve`.

Note como os argumentos de saída da sub-rotina (`blended_pp`, `z`) devem ser inicializados antes do `CALL`. A reserva VP varia apólice a apólice porque é conduzida pela perda incorrida real de cada apólice — apólices sem sinistros carregam reserva zero, então seu `reserve_pv` também é zero.


In [4]:
DADOS rated;
    DEFINIR portfolio;

    /* 1. Experiencia de perda propria da apolice como um premio puro */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. Pondera por credibilidade a experiencia propria contra a tarifa de classe.
          k = 4 anos-exposicao para credibilidade quase plena. */
    blended_pp = .;
    z = .;
    CHAMAR blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. Converte o custo de perda combinado (por carro-ano) em premio cobrado */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. Reserva de sinistro em aberto = 35% da perda incorrida, esperada
          para liquidar em 3 anos; desconta a valor presente a 4%. */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    MANTER policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
EXECUTAR;

PROCEDIMENTO IMPRIMIR DADOS=rated(obs=10) noobs RÓTULO;
    RÓTULO policy_id="ID da Apolice"
          region="Regiao"
          years_insured="Anos Segurado"
          exposure="Exposicao (carro-anos)"
          own_pp="Premio Puro Proprio"
          blended_pp="Premio Puro Combinado"
          z="Fator de Credibilidade (Z)"
          premium="Premio Cobrado"
          reserve_pv="Reserva a Valor Presente";
    TÍTULO "Carteira Tarifada (10 Primeiras Apolices)";
    VARIÁVEL policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
EXECUTAR;


                                       Carteira Tarifada (10 Primeiras Apolices)                                        

ID da Apolice     Regiao  Anos Segurado  Exposicao (carro-anos)  Premio Puro Proprio  Premio Puro Combinado  Fator de Credibilidade (Z)  Premio Cobrado  Reserva a Valor Presente
        10001  rural                  5                    1.36                    0                  91.67                       0.745          186.18                         0
        10002  urbano                 8                    0.97              3538.43                3039.59                       0.816         4402.95                   1067.95
        10003  urbano                 2                    1.53              4677.07                3046.88                       0.577         6961.51                   2226.55
        10004  suburbano              9                     2.4                    0                  90.69                       0.832          325.03               


NOTE: DATA rated


NOTE: Read 100 rows from portfolio.
NOTE: Wrote rated (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## Etapa 5 — Resumir o livro tarifado

Com cada apólice precificada através da mesma biblioteca compilada, podemos consolidar a carteira por região. Reportamos o prêmio médio cobrado, o fator médio de credibilidade, a perda total incorrida e a reserva total de sinistro a valor presente, e então derivamos o índice de sinistralidade implícito por segmento para ver se o prêmio carregado cobre a perda esperada.


In [5]:
PROCEDIMENTO MÉDIAS DADOS=rated mean sum maxdec=2 nonobs;
    CLASSE region;
    VARIÁVEL premium z incurred_loss reserve_pv;
    RÓTULO region="Regiao"
          premium="Premio Cobrado"
          z="Fator de Credibilidade (Z)"
          incurred_loss="Perda Incorrida"
          reserve_pv="Reserva a Valor Presente";
    TÍTULO "Resumo da Carteira por Regiao de Classificacao";
EXECUTAR;

/* Indice de sinistralidade implicito = perda incorrida / premio cobrado, mais a
   reserva a valor presente que o segmento carrega, por regiao. */
PROCEDIMENTO SQL;
    TÍTULO "Indice de Sinistralidade Implicito e Reserva VP por Regiao";
    SELECIONAR region,
           count(*)                              AS numero_apolices,
           sum(incurred_loss)                    AS perda_total     FORMATO=dollar12.2,
           sum(premium)                          AS premio_total  FORMATO=dollar12.2,
           sum(incurred_loss) / sum(premium)     AS indice_sinistralidade     FORMATO=percent8.1,
           sum(reserve_pv)                       AS reserva_vp_total FORMATO=dollar12.2
    DE_TABELA rated
    GROUP POR region
    ORDER POR region;
QUIT;
TÍTULO;


                                     Resumo da Carteira por Regiao de Classificacao                                     

                                                  The MEANS Procedure

                                       Analysis Variable : premium Premio Cobrado

        Regiao              Mean            Sum
        ---------------------------------------
        rural             476.61       11438.62
        suburbano         813.04       34147.74
        urbano           1987.17       67563.93
        ---------------------------------------

                                    Analysis Variable : z Fator de Credibilidade (Z)

        Regiao              Mean            Sum
        ---------------------------------------
        rural               0.71          17.14
        suburbano           0.68          28.36
        urbano              0.70          23.90
        ---------------------------------------

                                   Analysis Variable : incur


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## Interpretando os resultados

O DATA step de tarifação nunca escreve explicitamente uma única fórmula de desconto ou credibilidade embutida — ele apenas chama `pure_premium`, `blend_premium`, `charged_premium` e `pv_reserve`. Esse é o retorno do **PROC FCMP**: as premissas atuariais vivem em uma biblioteca compilada que pode ser testada unitariamente, controlada por versão e reutilizada em trabalhos de precificação, reservas e relatórios. Mudar a constante de credibilidade `k`, o carregamento de risco ou a taxa de despesa é uma única edição na biblioteca, não uma caçada por todo programa.

Lendo o livro tarifado e a consolidação regional:

- **Credibilidade (`z`)** cresce com `years_insured`, exatamente como a fórmula de flutuação limitada `Z = sqrt(n / (n + k))` determina. Entre as dez primeiras apólices, a apólice suburbana de um ano (10006) obtém `z = 0,447`, a apólice urbana de dois anos (10003) `z = 0,577`, enquanto a apólice suburbana de nove anos (10004) alcança `z = 0,832`. Apólices com pouca experiência são puxadas em direção à tarifa de classe regional; as de longa permanência se apoiam nas próprias perdas.
- **Combinação em ação:** apólices sem sinistros (a maior parte do livro) têm `own_pp = 0`, então `blend_premium` retorna um `blended_pp` próximo de `(1 - z)` vezes a tarifa de classe — por exemplo, a apólice 10001 (rural, `z = 0,745`) fica em `91,67` contra uma tarifa de classe de `360`/carro-ano. As duas apólices urbanas que tiveram sinistros, 10002 e 10003, em vez disso puxam seu custo de perda combinado para cima, em direção à sua própria experiência elevada (`3039,59` e `3046,88`).
- **Prêmio cobrado** fica acima do custo de perda combinado porque `charged_premium` adiciona um carregamento de risco de 12% e majora por uma taxa de despesa de 25%, um multiplicador fixo de `1,12 / 0,75 = 1,493` sobre o custo de perda. O prêmio médio cobrado fica em `476,61` (rural), `813,04` (suburbano) e `1.987,17` (urbano).
- **Reservas descontadas:** `pv_reserve` desconta a reserva de sinistro em aberto de cada apólice (35% da perda incorrida) por três anos a 4%, um fator de valor presente de `0,889`. Apólices sem sinistros carregam `reserve_pv = 0`; os dois sinistrados urbanos contribuem com `1.067,95` e `2.226,55`. Consolidado, o livro mantém uma reserva a valor presente de `R$ 2.151,56` (rural), `R$ 5.932,52` (suburbano) e `R$ 13.364,13` (urbano).
- **Índices de sinistralidade implícitos** ficam em 60,5% (rural), 55,8% (suburbano) e 63,6% (urbano) — todos confortavelmente abaixo de 100%, então o prêmio carregado cobre a perda esperada em todo segmento. O segmento urbano é o mais quente, impulsionado por seus dois grandes sinistros simulados; uma revisão de tarifa real testaria se esse sinal persiste ao longo de mais anos de acidentes antes de ajustar a tarifa manual.

A sub-rotina `blend_premium` também demonstra o idioma `OUTARGS` para retornar múltiplos resultados de um único `CALL` — o custo de perda combinado e o fator de credibilidade `z` voltam juntos — evitando chamadas de função separadas e mantendo a lógica de tarifação por apólice compacta e auditável.
